In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import PySimpleGUI as sg
import pyabf
import statistics 
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import os
from tkinter import Tk, filedialog

PySimpleGUI is now located on a private PyPI server.  Please add to your pip command: -i https://PySimpleGUI.net/install

The version you just installed should uninstalled:
   python -m pip uninstall PySimpleGUI
   python -m pip cache purge

Then install the latest from the private server:
python -m pip install --upgrade --extra-index-url https://PySimpleGUI.net/install PySimpleGUI

You can also force a reinstall using this command and it'll install the latest regardless of what you have installed currently
python -m pip install --force-reinstall --extra-index-url https://PySimpleGUI.net/install PySimpleGUI

Use python3 command if you're running on the Mac or Linux


In [2]:
def channel (file, search_ch): #gives the time period in every channel to search in
    channels_num = int(file.channelCount)
    
    if channels_num >=2: 
        my_file.setSweep(sweepNumber = 0, channel = 0) #Open 1st sweep to find the stimulus
        stim_line_ch0 = my_file.sweepC #will show only the stimulus (current) where baseline == 0
        stim_step_ch0 = np.array(np.nonzero(stim_line_ch0))[0] # everything what is not 0, is the stim period
        stim_start_ch0 = stim_step_ch0[0] #stim begin
        stim_end_ch0 = stim_step_ch0[-1] #stim stop

    if channels_num == 4: 
        my_file.setSweep(sweepNumber = 0, channel = 1) #channel index is a bog confusion. But sweepC in Ch1 returnes the stim line
        stim_line_ch2 = my_file.sweepC
        stim_step_ch2 = np.array(np.nonzero(stim_line_ch2))[0]
        stim_start_ch2 = stim_step_ch2[0]
        stim_end_ch2 = stim_step_ch2[-1]
        
    points_needed_pre = int(0.125*sample_rate) #calculate the mean value in 125 ms at baseline
    points_needed_in = int(0.25*sample_rate) #calculate the mean value in 250 ms during activity
    points_needed_after = int(0.5*sample_rate) #calculate the mean value in 1 s after activity

    if search_ch == 0:
        start = stim_start_ch0
        stop = stim_end_ch0
        pre_start = stim_start_ch0-points_needed_pre
        post_stop = stim_end_ch0+points_needed_after
        within_point = stim_end_ch0-points_needed_in

    if search_ch == 2:
        start = stim_start_ch2
        stop =  stim_end_ch2
        pre_start = stim_start_ch2-points_needed_pre
        post_stop = stim_end_ch2+points_needed_after
        within_point = stim_end_ch2-points_needed_in
        
    return start, stop, pre_start, post_stop, within_point

def sweep_4Aps (file, search_ch, start, stop, v_level= -10): #choose the sweep with at least 4 spikes
    #v_level gives the threashold for spike detection
    for sweep_num in range(len(file.sweepList)):
        file.setSweep(sweep_num, channel = search_ch)
        peaks = find_peaks(file.sweepY[start:stop], height=v_level)
        #print(peaks, 'My peaks')
        if len(peaks[0])>=4:
            sweep = sweep_num #4 sweeps were detected
            break
        elif len(peaks[0])<4 and sweep_num==len(file.sweepList)-1:
            print('Sweep is not detected!')
            sweep = np.nan
    return sweep

#find the last sweep with no APs    
def sweep_1AP (my_file_abf, search_ch, start, stop, v_level= -10):
    #choose the sweep with at least 1 spikes
    for sweep_num in range(len(my_file_abf.sweepList)):
        my_file_abf.setSweep(sweep_num, channel = search_ch)
        peaks = find_peaks(my_file_abf.sweepY[start:stop], height=v_level)
        if len(peaks[0])>=1:
            sweep = sweep_num -1 #sweep with 1 spike is detected but we use the one before
            break
        elif len(peaks[0])<1 and sweep_num==len(my_file_abf.sweepList):
            #print('Sweep is not detected!')
            #sweep = None
            sweep = sweep_num #the last sweep is anyways without a single spike
    return sweep

def infl_points (file, sweep, search_ch, start, stop): #inflection points detection, 
    #number of points for smoothing in gaus filter. Less than 5 gives more than 2 inflection points in AP
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        smooth = gaussian_filter1d(file.sweepY[start:stop], 3) # smooth
        smooth_d2 = np.gradient(np.gradient(smooth)) # compute second derivative
        infls = np.where(np.diff(np.sign(smooth_d2)))[0]# find switching points
        ind_infls = infls + start
        return ind_infls #else return None
    return [] 

def peak (file, sweep, search_ch, start, stop, v_level= -10): #find indexes of peaks 
    #v_level gives the threashold for spike detection
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        peaks= find_peaks(file.sweepY[start:stop], height=v_level, distance=10)
        ind_peaks = peaks[0] + start
        return ind_peaks #else return None
    return [] 

def spike_begin(ind_peaks, ind_infls):
    start_ap_ind = []
    end_ap_ind = []
    
    if len(ind_peaks) == 0 or len(ind_infls) == 0:
        return start_ap_ind, end_ap_ind

    for peak in ind_peaks: 

        infl_before_peak = [i for i in ind_infls if i < peak] # Найти индексы точек перегиба, которые меньше пика

        if len(infl_before_peak) >= 2: # Берём ПРЕДпоследнюю перед пиком
            start_ap_ind.append(infl_before_peak[-2])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            start_ap_ind.append(np.nan)

        infl_after_peak = [i for i in ind_infls if i > peak]

        if len(infl_after_peak) >= 2: # Берём вторую после пика пиком, в идеале дб в минимуме?
            end_ap_ind.append(infl_after_peak[1])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            end_ap_ind.append(np.nan)

    return start_ap_ind, end_ap_ind

def fwhm_1st_spike (file, sweep, search_ch, ap_start, ap_end, ap_peak, sampling_rate, height = 0.5): #fwhm calculation 
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        fwhm_full = scipy.signal.peak_widths(my_file.sweepY[ap_start:ap_end], #I give here info only about 1 spike (peak) !
                                            [ap_peak-ap_start], 
                                            rel_height=1 - height) #width [0] is in number of sampling points
        fwhm_full_time = np.round((fwhm_full[0][0]/sampling_rate)*1000, 3) #from sampling points to time in ms
        fwhm_indices = fwhm_full[2:4] + ap_start
        fwhm_height = fwhm_full[1][0]
        
        return fwhm_full_time, fwhm_indices, fwhm_height
    return np.nan, [], np.nan

def peak_amplitudes_ratio (file, sweep, search_ch, ap_start, ap_peak): #AP ratio calculation (Currently only 1st and 2nd)
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        amplitudes = [file.sweepY[peak] - file.sweepY[start] for peak, start in zip(ap_peak, ap_start)]

        # Отношение 2/1 (если доступно)
        ampl_ratio_21 = np.round(amplitudes[1] / amplitudes[0], 2) if len(amplitudes) >= 2 else np.nan

        # Список отношений i+1 / i начиная с 2 (т.е. 3/2, 4/3 и т.д.)
        successive_ratios = [np.round(amplitudes[i+1] / amplitudes[i], 2) 
                             for i in range(1, len(amplitudes)-1)]

        # Среднее значение этих отношений
        mean_successive_ratio = np.round(np.mean(successive_ratios), 2) if successive_ratios else np.nan

        return ampl_ratio_21, successive_ratios, mean_successive_ratio
    return np.nan, [], np.nan

def frequencies (ap_peak, sampling_rate):
    if len(ap_peak) == 0:
        return np.nan, np.nan, np.nan, np.nan
    freq_12 = np.round(sampling_rate/(ap_peak[1]-ap_peak[0]), 2) #Initial firing frequency Hz
    freq_late = np.round(sampling_rate/(ap_peak[-1]-ap_peak[-2]), 2) #Late firing frequency Hz
    freq_list = [sampling_rate/(ap_peak[i+1]-ap_peak[i]) for i in range (1, len(ap_peak)-1)]
    mean_freq = np.round(statistics.mean(freq_list), 2)#Mean firing frequency Hz
    ratio_late_12 = np.round(freq_late/freq_12, 2)
    return freq_12, freq_late, mean_freq, ratio_late_12

def delay_1st_spike (start, ap_peak, sampling_rate): # TODO maybe no need
    if len(ap_peak) == 0:
        return np.nan
    time_pre1ap = np.round(((ap_peak[0]-start)/sampling_rate)*1000, 2)
    return time_pre1ap

def IV_Rin(my_file_abf, #relative values of I and V and Rin
           voltage_channel, current_channel,
           pre_start, pre_end, 
           post_start, post_end, 
           voltages_list = [], currents_list = [],
           currents_delta_list = [], voltages_delta_list = [], 
           voltage_range_ind_list = [],
           Rin_relative_list = [], Rin_abs_list = []):
    
    my_sweep = sweep_1AP(my_file_abf, voltage_channel, pre_end, post_end)
    
    voltages_list.clear()
    currents_list.clear()
    currents_delta_list.clear()
    voltages_delta_list.clear()
    voltage_range_ind_list.clear()

    for sweep_index, my_sweepNumber in enumerate(range(my_sweep)):
        
        my_file_abf.setSweep(sweepNumber = my_sweepNumber, channel = voltage_channel)
            
        voltage_pre_stim = np.mean(my_file_abf.sweepY[pre_start:pre_end])
        voltage_dur_stim = np.mean(my_file_abf.sweepY[post_start:post_end])
            
        voltage_delta = voltage_dur_stim - voltage_pre_stim
            
        voltages_delta_list.append(voltage_delta)
        voltages_list.append(voltage_dur_stim)

        if voltage_dur_stim >= -85 and voltage_dur_stim <= -50: #to stay only in linear part
            voltage_range_ind_list.append(sweep_index)
            
        my_file_abf.setSweep(sweepNumber = my_sweepNumber, channel = current_channel)
            
        current_pre_stim = np.mean(my_file_abf.sweepY[pre_start:pre_end])
        current_dur_stim = np.mean(my_file_abf.sweepY[post_start:post_end])
                
        current_delta = current_dur_stim - current_pre_stim
        currents_delta_list.append(current_delta) 
        currents_list.append(current_dur_stim)
        
    Rin_relative_list = []
 
    if voltages_delta_list and currents_delta_list:
        Rin_relative_list = [x / y if abs(y) > 0.15 else np.nan for x, y in zip(voltages_delta_list, currents_delta_list)]
    
    if voltage_range_ind_list:       # the mean relative resistance
        Rin_from_range = [Rin_relative_list[x] for x in voltage_range_ind_list] #only resistances where V is in the range
        Rin_rel = np.round((np.nanmean(Rin_from_range))*1000, 3)
    else:
        Rin_rel = np.nan
        
    slope = np.nan
    intercept = np.nan 
    
    currents_list_ar = np.array(currents_list)
    voltages_list_ar = np.array(voltages_list)
    
    #if len(currents_list_ar) >= 2 and len(voltages_list_ar) >= 2: 
    if len(voltage_range_ind_list) >= 2: 
        (p, residuals, rank, singular_values, rcond) = np.polyfit(currents_list_ar[voltage_range_ind_list], 
                                  voltages_list_ar[voltage_range_ind_list], 1, full = True) #linear fit
        slope, intercept = p
        x_fit = np.linspace(min(currents_list_ar[voltage_range_ind_list]), max(currents_list_ar[voltage_range_ind_list]), 100)
        y_fit = slope * x_fit + intercept
        
        if residuals.size > 0:
            SEE = residuals[0]
            SST = np.sum((voltages_list_ar[voltage_range_ind_list] - np.mean(voltages_list_ar[voltage_range_ind_list])) ** 2)
            R2 = np.round(1 - SEE / SST, 3) if SST != 0 else np.nan
        else:
            R2 = np.nan  # Недостаточно точек для вычисления ошибки
        #plt.scatter(currents_list_ar, voltages_list_ar, 
              #  color='blue', label='Data (IV points)')
        #plt.plot(x_fit, y_fit, color='red', label=f'Linear Fit (Rin ≈ {np.round(slope, 2)} MΩ)')   
    
            
        Rin_abs = np.round(slope*1000,3) # the mean absolute resistance
        v_zero_current = np.round(intercept, 3)
    else:
        Rin_abs = np.nan
        v_zero_current = np.nan
        R2 = np.nan
    
    
                
    return currents_delta_list, voltages_delta_list, Rin_rel, currents_list, voltages_list, Rin_abs, v_zero_current, R2

def IV_values(file, sweep, voltage_channel, current_channel,
              pre_start, pre_stop, 
              in_start, in_stop):
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = voltage_channel)
        voltage_pre_stim = np.round(np.mean(file.sweepY[pre_start:pre_stop]), 3) #the cell was held at...
        file.setSweep(sweep, channel = current_channel)
        current_pre_stim = np.mean(file.sweepY[pre_start:pre_stop]) 
        current_in_stim = np.mean(file.sweepY[in_start:in_stop])
        current_injection = np.round(current_in_stim - current_pre_stim, 3) # the applied current where more than 4 APs
        return voltage_pre_stim, current_injection
    return np.nan, np.nan

In [3]:
folder_path = r'C:\Users\user\Documents\IMBIT\M1 M2 L1 Ch0' 

In [4]:
# Задаем пустую таблицу
# Определи названия столбцов
columns = ['File Name', 
           'AP2/1 ration', 
           'Attenuation trend', 
           'Initial Firing Frequency [0:1], Hz',
           'Late Frequency ALL [-2:-1], Hz', 
           'Mean Frequency [1:-1], Hz', 
           'Freq ratio', 
           'FWHM, ms', 
           'Delay AP1, ms', 
           'Rin_abs, MOhms', 
           'Rin_rel, MOhms', 
           'Vrest mV', 
           'cell at mV',
           'inj_current, pA',
           'R2 for abs Rin'
          ]



# Создай пустой DataFrame с этими столбцами (без данных)
df = pd.DataFrame(columns=columns)

# Сохрани в Excel
excel_path = f'{folder_path}\\results_depolarisation.xlsx'
df.to_excel(excel_path, index=False)

print("Пустая таблица создана: results_depolarisation.xlsx")

Пустая таблица создана: results_depolarisation.xlsx


In [5]:
my_search_ch = 0

In [6]:
# Выбираем папку!
root = Tk()
root.withdraw()
root.attributes('-topmost', True)

# Ask the user to select a folder
folder_path_all = filedialog.askdirectory(
    title="Выберите папку"
)

print("Выбрана папка:", folder_path_all)

Выбрана папка: F:/M1 M2 L1/Ch0_2


In [7]:
df = pd.read_excel(excel_path)
new_rows = []

for file_name in os.listdir(folder_path_all):
    if file_name.endswith(".abf"):
        file_path = os.path.join(folder_path_all, file_name)
        
        try:

            # Открываем ABF-файл
            my_file = pyabf.ABF(file_path)
            filename = my_file.abfID  # без расширения .abf


            print(f"Обрабатывается файл: {filename}")

            channels_num = int(my_file.channelCount)
            sample_rate = int(my_file.dataRate)

            my_start, my_stop, my_pre_start, my_post_stop, my_within_point = channel(my_file, my_search_ch) #returns stimulation start and stop in the chosen channel

            my_sweep = sweep_4Aps(my_file, my_search_ch, my_start, my_stop) # find the sweep with 4 spikes

            my_ind_infls = infl_points (my_file, my_sweep, my_search_ch, my_start, my_stop) #find all inflection points

            voltage_level = np.mean(my_file.sweepY[my_ind_infls]) #TODO

            my_ind_peaks = peak (my_file, my_sweep, my_search_ch, my_start, my_stop) # find peaks' indexes

            my_start_ap_ind, my_end_ap_ind = spike_begin(my_ind_peaks, my_ind_infls) # Find start and end of spikes


            my_fwhm, fwhm_ind, my_fwhm_height  = fwhm_1st_spike (my_file, my_sweep, my_search_ch, # find fwhm
                                                my_start_ap_ind[0], my_end_ap_ind[0], my_ind_peaks[0], 
                                                sample_rate)

            ap_ampl_ratio, my_successive_ratios, my_mean_successive_ratio = peak_amplitudes_ratio (my_file, my_sweep, my_search_ch, 
                                                                                                   my_start_ap_ind, my_ind_peaks) #ampl ratio

            in_freq, late_freq, mean_freq, ratio_freq_late_init = frequencies (my_ind_peaks, sample_rate) #all frequencies

            #my_data_check = scipy.signal.peak_prominences(my_file.sweepY[my_start_ap_ind[0]:my_end_ap_ind[0]], my_ind_peaks[:1]-my_start_ap_ind[0])

            time_pre1ap = delay_1st_spike (my_start, my_ind_peaks, sample_rate) #2st spike delay

            IVRin_params = IV_Rin(my_file, my_search_ch, my_search_ch + 1, my_pre_start, my_start, my_within_point, my_stop)
            my_Rin_rel, my_Rin_abs, resting_potential, m_R2 = IVRin_params[2],IVRin_params[5], IVRin_params[-2], IVRin_params[-1]

            stim_params = IV_values(my_file, my_sweep, my_search_ch, my_search_ch + 1, 
                                             my_pre_start, my_start, my_within_point, my_stop)
            hold_voltage, inj_current = stim_params [0], stim_params [1]


            my_file.setSweep(sweepNumber = my_sweep, channel = my_search_ch)

            plt.figure(figsize=(40,20))
            plt.suptitle(f"Depolarisation: {filename}", fontsize=20)
            #The main signal
            plt.plot(my_file.sweepX[my_pre_start:my_post_stop], my_file.sweepY[my_pre_start:my_post_stop], label='Noisy Data', linewidth=1)
            #Peaks
            plt.scatter(my_file.sweepX[my_ind_peaks], my_file.sweepY[my_ind_peaks])
            #Begining of the spike
            plt.scatter(my_file.sweepX[my_start_ap_ind], my_file.sweepY[my_start_ap_ind])
            plt.vlines(my_start/sample_rate, -50, 0, linestyle='dotted', linewidth=0.5, label='Start')
            plt.vlines(my_stop/sample_rate, -50, 0, linestyle='dotted', linewidth=0.5, label='Stop')

            time = np.arange(len(my_file.sweepY)) / sample_rate * 1000  # convert to ms

            plt.hlines(y=my_fwhm_height,
                        xmin=(fwhm_ind[0] / sample_rate),
                        xmax=(fwhm_ind[1] / sample_rate),
                        color='red')

            save_path = os.path.join(folder_path, filename)
            plt.savefig(save_path)
            plt.close()

            #new_values = 
            new_rows.append([filename, 
                          ap_ampl_ratio, 
                          my_mean_successive_ratio,
                          in_freq,
                          late_freq,
                          mean_freq,
                          ratio_freq_late_init,
                          my_fwhm,
                          time_pre1ap,
                          my_Rin_abs,
                          my_Rin_rel,
                          resting_potential,
                          hold_voltage,
                          inj_current,
                          m_R2
                            ])

            #df = pd.read_excel(excel_path)
            #new_row = pd.Series(new_values, index=df.columns)
            #df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
            #df.to_excel(excel_path, index=False)
            
        except Exception as e:
                print(f"Ошибка при обработке файла {file_name}: {e}")
                continue  # перейти к следующему файлу
                            
                            
# 4. После цикла: превращаем new_rows в DataFrame и объединяем с df
df_new = pd.DataFrame(new_rows, columns=df.columns)
df = pd.concat([df, df_new], ignore_index=True)

# 5. Один раз сохраняем Excel
df.to_excel(excel_path, index=False)

Обрабатывается файл: 2025_04_14_0024
Обрабатывается файл: 2025_04_14_0025
Обрабатывается файл: 2025_04_14_0027
Обрабатывается файл: 2025_04_17_0010
Обрабатывается файл: 2025_04_17_0018
Обрабатывается файл: 2025_04_17_0020
Обрабатывается файл: 2025_04_17_0024
Обрабатывается файл: 2025_05_09_0004
Обрабатывается файл: 2025_05_09_0006
Обрабатывается файл: 2025_05_20_0000
Обрабатывается файл: 2025_05_22_0001
Обрабатывается файл: 2025_05_22_0008
Обрабатывается файл: 2025_05_22_0013
Обрабатывается файл: 2025_05_22_0015
Обрабатывается файл: 2025_05_22_0016
Обрабатывается файл: 2025_05_22_0018
Обрабатывается файл: 2025_05_22_0021
Обрабатывается файл: 2025_05_23_0000
Обрабатывается файл: 2025_05_23_0003
Обрабатывается файл: 2025_05_23_0004
